# Complete End-to-End LADDER Pipeline

This notebook implements the complete 3-step LADDER pipeline:
- **Step 3a**: Binary sentence classification (attack vs non-attack)
- **Step 3b**: Attack phrase extraction (NER/sequence tagging)
- **Step 4**: MITRE ATT&CK technique mapping

Data flows through each step with progressive filtering:
```
Raw Text → Step 3a → Attack Sentences → Step 3b → Attack Phrases → Step 4 → MITRE IDs
```

## 📦 Section 1: Installation and Imports

In [4]:
# Install required packages
!pip install transformers torch datasets sentence-transformers scipy pandas nltk scikit-learn --quiet
import nltk
nltk.download('punkt', quiet=True)
print("✓ All packages installed successfully!")

✓ All packages installed successfully!


In [5]:
# Import all required libraries
import os
import json
import torch
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

# Transformers
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoTokenizer,
    pipeline
)

# Sentence transformers for Step 4
from sentence_transformers import SentenceTransformer
from scipy import spatial

# NLP utilities
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful!
PyTorch version: 2.9.1+cu128
CUDA available: True


## ⚙️ Section 2: Configuration

Update these paths to match your trained models and data files.

In [52]:
@dataclass
class PipelineConfig:
    """Configuration for the complete LADDER pipeline"""
    
    # ========== STEP 3a: Sentence Classification ==========
    step3a_model_name: str = "roberta-large"  # Base model used for training
    step3a_weights_path: str = "./runs/ladder_cls/weights.pt"  # Trained weights
    step3a_modelpath: str = "./runs/ladder_cls/full_model"
    step3a_confidence_threshold: float = 0.5  # Minimum confidence for positive class
    
    # ========== STEP 3b: Attack Phrase Extraction ==========
    step3b_model_path: str = "./outputs_ner/checkpoint-1080"  # Best checkpoint directory
    step3b_max_length: int = 128  # Maximum sequence length
    step3b_filter_verbs: bool = True  # Post-processing: require verbs in phrases
    
    # ========== STEP 4: MITRE Mapping ==========
    mitre_csv_path: str = "/home/jovyan/work/MITREMapping/enterprise-techniques-sub.csv"  # MITRE ATT&CK framework
    mitre_threshold: float = 0.5  # Distance threshold (LADDER optimal: 0.6)
    mitre_weight_title: float = 0.4  # Title weight (LADDER optimal: 0.4)
    sentence_model_name: str = "all-mpnet-base-v2"  # Sentence transformer model
    
    # ========== General ==========
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size: int = 16  # For batch processing
    verbose: bool = True  # Print progress messages

# Create configuration
config = PipelineConfig()

print("Configuration:")
print(f"  Device: {config.device}")
print(f"  Step 3a model: {config.step3a_model_name}")
print(f"  Step 3b model: {config.step3b_model_path}")
print(f"  MITRE threshold: {config.mitre_threshold}")
print(f"  MITRE weight: {config.mitre_weight_title}")

Configuration:
  Device: cuda
  Step 3a model: roberta-large
  Step 3b model: ./outputs_ner/checkpoint-1080
  MITRE threshold: 0.5
  MITRE weight: 0.4


## 🔧 Section 3: Load All Models

Load trained models for each step of the pipeline.

### 3.1: Load Step 3a Model (Sentence Classification)

In [7]:
print("Loading Step 3a: Sentence Classification Model...")
print("=" * 60)

step3a_model = AutoModelForSequenceClassification.from_pretrained(
    config.step3a_modelpath
)
step3a_tokenizer = AutoTokenizer.from_pretrained(
    config.step3a_modelpath
)

step3a_model.to(config.device)
step3a_model.eval()

step3a_classifier = pipeline(
    "text-classification",
    model=step3a_model,
    tokenizer=step3a_tokenizer,
    device=0 if config.device == "cuda" else -1,
    return_all_scores=True,
    truncation=True
)

print(f"✓ Model loaded from: {config.step3a_modelpath}")
print(f"  Device: {config.device}")

Loading Step 3a: Sentence Classification Model...


Loading weights: 100%|█████████████████████| 393/393 [00:01<00:00, 247.20it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]


✓ Model loaded from: ./runs/ladder_cls/full_model
  Device: cuda


### 3.2: Load Step 3b Model (Attack Phrase Extraction)

In [8]:
print("Loading Step 3b: Attack Phrase Extraction Model...")
print("=" * 60)

# Load model and tokenizer
if os.path.exists(config.step3b_model_path):
    step3b_model = AutoModelForTokenClassification.from_pretrained(config.step3b_model_path)
    step3b_tokenizer = AutoTokenizer.from_pretrained(config.step3b_model_path)
    print(f"✓ Loaded model from: {config.step3b_model_path}")
else:
    print(f"⚠️  Warning: Model not found at {config.step3b_model_path}")
    print(f"   Loading base model (for demo purposes only)")
    step3b_model = AutoModelForTokenClassification.from_pretrained(
        "bert-base-cased",
        num_labels=2,
        id2label={0: 'ATK', 1: 'O'},
        label2id={'ATK': 0, 'O': 1}
    )
    step3b_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Move to device
step3b_model.to(config.device)
step3b_model.eval()

# Get label mappings
step3b_id2label = step3b_model.config.id2label
step3b_label2id = step3b_model.config.label2id

print(f"✓ Step 3b model loaded and ready!")
print(f"  Labels: {list(step3b_id2label.values())}")
print(f"  Device: {config.device}")
print()

Loading Step 3b: Attack Phrase Extraction Model...


Loading weights: 100%|█████████████████████| 391/391 [00:01<00:00, 251.87it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]


✓ Loaded model from: ./outputs_ner/checkpoint-1080
✓ Step 3b model loaded and ready!
  Labels: ['ATK', 'O']
  Device: cuda



### 3.3: Load Step 4 Model (MITRE Mapping)

In [37]:
def load_mitre_framework_subtech(csv_path: str) -> Dict:
    """
    Load MITRE ATT&CK sub-techniques from enterprise-techniques-sub.csv.
    
    Returns dict with:
        key = full_id (e.g., "T1059.001")
        value = {id, tech_id, sub_id, node_name, node_desc, combo_title, combo_desc}
    """
    df = pd.read_csv(csv_path)
    
    print(f"Loading sub-techniques from: {csv_path}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Total rows: {len(df)}")
    
    # Ensure expected columns exist
    required_cols = ["ID", "SubTechID", "Name", "Description"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Expected column '{col}' not found in CSV")
    
    # Forward-fill technique ID so sub-tech rows inherit parent Txxxx
    df["ID"] = df["ID"].ffill()
    
    # Keep only rows that are actual sub-techniques (SubTechID not NaN)
    sub_df = df[df["SubTechID"].notna()].copy()
    print(f"  Sub-techniques: {len(sub_df)}")
    
    subtech_dict = {}
    
    for _, row in sub_df.iterrows():
        tech_id = str(row["ID"]).strip()          # e.g., "T1059"
        sub_id = row["SubTechID"]                 # e.g., 0.001
        name = str(row["Name"])                   # e.g., "PowerShell"
        desc = str(row["Description"])            # e.g., "Adversaries may abuse..."
        
        # Convert SubTechID (0.001) to suffix ("001")
        try:
            sub_suffix = f"{float(sub_id):.3f}".split(".")[1]
        except Exception:
            sub_suffix = str(sub_id).split(".")[-1]
        
        # Build full ID: T1059.001
        full_id = f"{tech_id}.{sub_suffix}"
        
        # Create combined title and description for embedding
        combo_title = f"{tech_id} :: {name}"
        combo_desc = f"{tech_id} - {name}\\n\\n{sub_id} - {name}\\n{desc}"
        
        subtech_dict[full_id] = {
            "id": full_id,
            "tech_id": tech_id,
            "sub_id": sub_suffix,
            "node_name": name,
            "node_desc": desc,
            "combo_title": combo_title,
            "combo_desc": combo_desc
        }
    
    print(f"✓ Loaded {len(subtech_dict)} sub-techniques")
    return subtech_dict


In [38]:
print("Loading Step 4: MITRE ATT&CK Mapping Model (Sub-Technique Level)...")
print("=" * 60)
# Embedding cache for query phrases
embedding_cache = {}

# Load sentence transformer
step4_model = SentenceTransformer(config.sentence_model_name)
print(f"✓ Loaded sentence transformer: {config.sentence_model_name}")

# Load MITRE ATT&CK sub-techniques
mitre_subtech_dict = load_mitre_framework_subtech(config.mitre_csv_path)
print(f"✓ Loaded {len(mitre_subtech_dict)} MITRE sub-techniques")

# Build embeddings for all sub-techniques
mitre_subtech_embeddings = build_subtech_embeddings(mitre_subtech_dict)

# Embedding cache for query phrases
embedding_cache = {}

print(f"✓ Step 4 ready!")
print()

Loading Step 4: MITRE ATT&CK Mapping Model (Sub-Technique Level)...


Loading weights: 100%|██████████████████████████████████████████████| 199/199 [00:00<00:00, 593.08it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Loaded sentence transformer: all-mpnet-base-v2
Loading sub-techniques from: /home/jovyan/work/MITREMapping/enterprise-techniques-sub.csv
  Columns: ['ID', 'SubTechID', 'Name', 'Description']
  Total rows: 576
  Sub-techniques: 385
✓ Loaded 385 sub-techniques
✓ Loaded 385 MITRE sub-techniques
Building embeddings for 385 sub-techniques...
✓ Built 385 sub-technique embeddings
✓ Step 4 ready!



## 🎯 Section 4: Pipeline Functions

Define functions for each step of the pipeline.

### 4.1: Step 3a Functions (Sentence Classification)

In [39]:
def classify_sentences(sentences: List[str], threshold: float = 0.5) -> List[Dict]:
    """
    Step 3a: Classify sentences as attack (1) or non-attack (0).
    
    Args:
        sentences: List of sentences to classify
        threshold: Confidence threshold for positive class
    
    Returns:
        List of dicts with sentence, label, and confidence
    """
    if not sentences:
        return []
    
    # Run classification
    results = step3a_classifier(sentences)
    
    classified = []
    for sent, preds in zip(sentences, results):
        # Get prediction for label '1' (attack)
        attack_pred = next((p for p in preds if p['label'] == '1'), None)
        
        if attack_pred and attack_pred['score'] >= threshold:
            classified.append({
                'sentence': sent,
                'label': '1',
                'confidence': attack_pred['score'],
                'is_attack': True
            })
        else:
            non_attack_pred = next((p for p in preds if p['label'] == '0'), None)
            classified.append({
                'sentence': sent,
                'label': '0',
                'confidence': non_attack_pred['score'] if non_attack_pred else 1.0 - attack_pred['score'],
                'is_attack': False
            })
    
    return classified
def classify_sentences(sentences, threshold):

    results = step3a_classifier(sentences)
    classified = []

    for sent, pred in zip(sentences, results):

        is_attack = (
            pred['label'] == '1'
            and pred['score'] >= threshold
        )

        classified.append({
            'sentence': sent,
            'label': pred['label'],
            'confidence': pred['score'],
            'is_attack': is_attack
        })

    return classified

def filter_attack_sentences(classified_results: List[Dict]) -> List[str]:
    """
    Filter to keep only attack sentences (label=1).
    
    Args:
        classified_results: Output from classify_sentences()
    
    Returns:
        List of attack sentences
    """
    return [r['sentence'] for r in classified_results if r['is_attack']]

print("✓ Step 3a functions defined")

✓ Step 3a functions defined


### 4.2: Step 3b Functions (Attack Phrase Extraction)

In [40]:
def extract_attack_phrases(sentence: str) -> List[Dict]:
    """
    Step 3b: Extract attack pattern phrases from a sentence using NER.
    
    Args:
        sentence: Input sentence
    
    Returns:
        List of dicts with phrase, tokens, and tags
    """
    if not sentence or not sentence.strip():
        return []
    
    # Tokenize sentence
    words = sentence.strip().split()
    
    # Encode
    encoding = step3b_tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=config.step3b_max_length
    )
    
    # ⚠️ IMPORTANT: Get word_ids BEFORE moving to device!
    word_ids = encoding.word_ids()
    
    # Now move to device
    encoding_tensors = {k: v.to(config.device) for k, v in encoding.items()}
    
    # Predict
    with torch.no_grad():
        outputs = step3b_model(**encoding_tensors)
        predictions = outputs.logits.argmax(-1)[0].tolist()
    
    # Align predictions with words using word_ids we saved earlier
    word_predictions = []
    used_words = set()
    
    for pred_id, word_id in zip(predictions, word_ids):
        if word_id is None or word_id in used_words:
            continue
        
        used_words.add(word_id)
        label = step3b_id2label[pred_id]
        word_predictions.append((words[word_id], label))
    
    # Extract contiguous ATK sequences
    phrases = []
    current_phrase = []
    current_tokens = []
    
    for word, label in word_predictions:
        if label == 'ATK':
            current_phrase.append(word)
            current_tokens.append(word)
        elif current_phrase:
            phrase_text = ' '.join(current_phrase)
            phrases.append({
                'phrase': phrase_text,
                'tokens': current_tokens.copy(),
                'sentence': sentence
            })
            current_phrase = []
            current_tokens = []
    
    # Don't forget last phrase
    if current_phrase:
        phrase_text = ' '.join(current_phrase)
        phrases.append({
            'phrase': phrase_text,
            'tokens': current_tokens.copy(),
            'sentence': sentence
        })
    
    return phrases

def has_verb(phrase: str) -> bool:
    """
    Check if phrase contains a verb (LADDER post-processing).
    
    Args:
        phrase: Text to check
    
    Returns:
        True if phrase contains a verb
    """
    try:
        tokens = nltk.word_tokenize(phrase)
        pos_tags = nltk.pos_tag(tokens)
        return any(tag.startswith('VB') for _, tag in pos_tags)
    except:
        return True  # If parsing fails, keep the phrase

def extract_phrases_from_sentences(sentences: List[str], filter_verbs: bool = True) -> List[Dict]:
    """
    Extract attack phrases from multiple sentences.
    
    Args:
        sentences: List of sentences
        filter_verbs: If True, keep only phrases with verbs
    
    Returns:
        List of phrase dicts
    """
    all_phrases = []
    
    for sent in sentences:
        phrases = extract_attack_phrases(sent)
        
        if filter_verbs:
            phrases = [p for p in phrases if has_verb(p['phrase'])]
        
        all_phrases.extend(phrases)
    
    return all_phrases

print("✓ Step 3b functions defined")

✓ Step 3b functions defined


### 4.3: Step 4 Functions (MITRE Mapping)

In [41]:
def get_embedding(text: str) -> np.ndarray:
    """
    Get sentence embedding with caching.
    
    Args:
        text: Input text
    
    Returns:
        Embedding vector
    """
    if text not in embedding_cache:
        embedding_cache[text] = step4_model.encode([text])[0]
    return embedding_cache[text]

def cosine_distance(emb1: np.ndarray, emb2: np.ndarray) -> float:
    """
    Compute cosine distance between two embeddings.
    
    Args:
        emb1: First embedding
        emb2: Second embedding
    
    Returns:
        Cosine distance (1 - cosine similarity)
    """
    return spatial.distance.cosine(emb1, emb2)

def map_to_mitre_subtech(
    phrase: str,
    subtech_embeddings: Dict,
    subtech_dict: Dict,
    threshold: float = 0.25
) -> Tuple[Optional[str], float, Optional[str], Optional[str]]:
    """
    Map phrase to MITRE sub-technique using cosine similarity.
    
    Args:
        phrase: Attack pattern phrase
        subtech_embeddings: Pre-computed embeddings for sub-techniques
        subtech_dict: Sub-technique metadata
        threshold: Maximum acceptable distance
    
    Returns:
        (subtech_id, distance, subtech_name, tech_id) or (None, distance, None, None)
    """
    query_emb = get_embedding(phrase)
    
    best_id = None
    best_dist = float('inf')
    
    # Find closest sub-technique
    for full_id, emb in subtech_embeddings.items():
        dist = cosine_distance(query_emb, emb)
        if dist < best_dist:
            best_dist = dist
            best_id = full_id
    
    # Check threshold
    if best_dist > threshold:
        return None, best_dist, None, None
    
    # Get metadata
    info = subtech_dict.get(best_id, {})
    subtech_name = info.get("node_name", "")
    tech_id = info.get("tech_id", "")
    
    return best_id, best_dist, subtech_name, tech_id

def map_phrases_to_mitre_subtech(
    phrases: List[Dict],
    subtech_embeddings: Dict,
    subtech_dict: Dict,
    threshold: float = 0.25
) -> List[Dict]:
    """Map multiple phrases to MITRE sub-techniques"""
    mapped = []
    
    for phrase_dict in phrases:
        phrase = phrase_dict['phrase']
        subtech_id, distance, subtech_name, tech_id = map_to_mitre_subtech(
            phrase, subtech_embeddings, subtech_dict, threshold
        )
        
        result = {
            'phrase': phrase,
            'sentence': phrase_dict['sentence'],
            'tokens': phrase_dict['tokens'],
            'subtech_id': subtech_id,           # ← NEW: T1059.001
            'subtech_name': subtech_name,       # ← NEW: PowerShell
            'technique_id': tech_id,            # ← Parent: T1059
            'distance': distance,
            'accepted': subtech_id is not None
        }
        
        mapped.append(result)
    
    return mapped


def build_subtech_embeddings(subtech_dict: Dict) -> Dict:
    """
    Build embeddings for all sub-techniques.
    
    For each sub-technique, creates a single embedding from:
        TechID + SubTechName + SubTechDescription
    
    Returns:
        Dict mapping full_id to embedding vector
    """
    embeddings = {}
    
    print(f"Building embeddings for {len(subtech_dict)} sub-techniques...")
    
    for full_id, info in subtech_dict.items():
        tech_id = info.get("tech_id", "")
        node_name = info.get("node_name", "")
        node_desc = info.get("node_desc", "")
        
        # Combine all text for richer embedding
        combo_text = f"{tech_id}. {node_name}. {node_desc}"
        
        embeddings[full_id] = get_embedding(combo_text)
    
    print(f"✓ Built {len(embeddings)} sub-technique embeddings")
    return embeddings

print("✓ Sub-technique embedding function defined")

print("✓ Step 4 functions defined")

✓ Sub-technique embedding function defined
✓ Step 4 functions defined


## 🚀 Section 5: Complete Pipeline Function

In [42]:
def split_sentences_nltk(text: str) -> List[str]:
    """
    Split text into sentences using NLTK's trained sentence tokenizer.
    
    This correctly handles:
    - File extensions (.exe, .dll, .sys)
    - Abbreviations (e.g., etc., Dr.)
    - URLs and domains
    - Decimal numbers
    
    Args:
        text: Input text
    
    Returns:
        List of sentences
    """
    sentences = nltk.sent_tokenize(text)
    return [s.strip() for s in sentences if s.strip()]


def process_text_through_pipeline(
    text: str,
    verbose: bool = True
) -> Dict:
    """
    Run complete LADDER pipeline on input text.
    
    Pipeline flow:
    1. Split text into sentences
    2. Step 3a: Classify sentences (attack vs non-attack)
    3. Step 3b: Extract attack phrases from attack sentences
    4. Step 4: Map phrases to MITRE ATT&CK techniques
    
    Args:
        text: Input text (can be multiple sentences)
        verbose: Print progress messages
    
    Returns:
        Dict containing results from each step
    """
    if verbose:
        print("\n" + "="*70)
        print("RUNNING COMPLETE LADDER PIPELINE")
        print("="*70)
    
    # Split into sentences (simple split for now)
    #sentences = [s.strip() for s in text.split('.') if s.strip()]

    sentences = split_sentences_nltk(text)
    for i, s in enumerate(sentences, 1):
        print(f"  {i}. {s}")
    print(f"  Total: {len(sentences)} sentences")
    
    if verbose:
        print(f"\n📄 Input: {len(sentences)} sentences")
        print("-" * 70)
    
    # ========== STEP 3a: Sentence Classification ==========
    if verbose:
        print("\n🔍 STEP 3a: Classifying sentences...")
    
    classified = classify_sentences(
        sentences,
        threshold=config.step3a_confidence_threshold
    )
    attack_sentences = filter_attack_sentences(classified)
    
    n_attack = len(attack_sentences)
    n_total = len(sentences)
    
    if verbose:
        print(f"  ✓ Identified {n_attack}/{n_total} attack sentences ({n_attack/n_total*100:.1f}%)")
        if n_attack > 0 and n_attack <= 5:
            for i, sent in enumerate(attack_sentences[:5], 1):
                print(f"    {i}. {sent[:80]}..." if len(sent) > 80 else f"    {i}. {sent}")
    
    # ========== STEP 3b: Attack Phrase Extraction ==========
    if verbose:
        print(f"\n🎯 STEP 3b: Extracting attack phrases...")
    
    phrases = extract_phrases_from_sentences(
        attack_sentences,
        filter_verbs=config.step3b_filter_verbs
    )
    
    if verbose:
        print(f"  ✓ Extracted {len(phrases)} attack phrases")
        if len(phrases) > 0 and len(phrases) <= 5:
            for i, p in enumerate(phrases[:5], 1):
                print(f"    {i}. {p['phrase']}")
    
    # ========== STEP 4: MITRE Sub-Technique Mapping ==========
    if verbose:
        print(f"\n🗺️  STEP 4: Mapping to MITRE ATT&CK (sub-technique level)...")
    
    mapped = map_phrases_to_mitre_subtech(
        phrases,
        mitre_subtech_embeddings,
        mitre_subtech_dict,
        threshold=config.mitre_threshold
    )
    accepted = [m for m in mapped if m['accepted']]
    
    if verbose:
        print(f"  ✓ Mapped {len(accepted)}/{len(phrases)} phrases to MITRE techniques")
    
    # Compile results
    results = {
        'input': {
            'text': text,
            'n_sentences': n_total
        },
        'step3a': {
            'classified': classified,
            'attack_sentences': attack_sentences,
            'n_attack': n_attack,
            'n_non_attack': n_total - n_attack
        },
        'step3b': {
            'phrases': phrases,
            'n_phrases': len(phrases)
        },
        'step4': {
            'mapped': mapped,
            'accepted': accepted,
            'n_mapped': len(accepted),
            'n_rejected': len(phrases) - len(accepted)
        }
    }
    
    if verbose:
        print("\n" + "="*70)
        print(f"✓ Pipeline complete!")
        print(f"  Input: {n_total} sentences")
        print(f"  Step 3a → {n_attack} attack sentences ({n_attack/n_total*100:.1f}%)")
        print(f"  Step 3b → {len(phrases)} attack phrases")
        print (phrases)
        print(f"  Step 4 → {len(accepted)} MITRE mappings")
        print("="*70 + "\n")
    
    return results

print("✓ Complete pipeline function defined")


✓ Complete pipeline function defined


## 📊 Section 6: Helper Functions for Results Display

In [43]:
def display_results(results: Dict, show_rejected: bool = False):
    """
    Display pipeline results in a formatted table.
    
    Args:
        results: Output from process_text_through_pipeline()
        show_rejected: If True, also show rejected mappings
    """
    print("\n" + "="*90)
    print("FINAL RESULTS: ATTACK PHRASES WITH MITRE ATT&CK MAPPINGS")
    print("="*90)
    
    accepted = results['step4']['accepted']
    
    if not accepted:
        print("\n⚠️  No attack phrases were successfully mapped to MITRE techniques.")
        if results['step3b']['n_phrases'] > 0:
            print(f"   {results['step3b']['n_phrases']} phrases were extracted but didn't match any technique.")
        return
    
    # Create DataFrame for nice display
    display_data = []
    for i, m in enumerate(accepted, 1):
        display_data.append({
            '#': i,
            'Attack Phrase': m['phrase'][:50] + '...' if len(m['phrase']) > 50 else m['phrase'],
            'Sub-Technique': m['subtech_id'],           # T1059.001
            'Name': m['subtech_name'][:30] + '...' if len(m['subtech_name']) > 30 else m['subtech_name'],
            'Parent Tech': m['technique_id'],           # T1059
            'Distance': f"{m['distance']:.3f}"
        })
    
    df = pd.DataFrame(display_data)
    print("\n" + df.to_string(index=False))
    
    # Show rejected mappings
    if show_rejected:
        rejected = [m for m in results['step4']['mapped'] if not m['accepted']]
        if rejected:
            print("\n" + "-"*90)
            print(f"REJECTED MAPPINGS (distance >= {config.mitre_threshold}):")
            print("-"*90)
            
            for i, m in enumerate(rejected, 1):
                print(f"\n{i}. Phrase: {m['phrase']}")
                print(f"   Best Match : {m['subtech_id']} - {m['subtech_name']}")
                print(f"   Parent Tech: {m['technique_id']}")
                print(f"   Distance   : {m['distance']:.3f} (threshold: {config.mitre_threshold})")
    
    print("\n" + "="*90 + "\n")

def export_results_to_csv(results: Dict, output_path: str = "ladder_output.csv"):
    """
    Export results to CSV file.
    
    Args:
        results: Output from process_text_through_pipeline()
        output_path: Path to save CSV
    """
    accepted = results['step4']['accepted']
    
    if not accepted:
        print("⚠️  No results to export.")
        return
    
    # Create DataFrame
    export_data = []
    for m in accepted:
        export_data.append({
            'attack_phrase': m['phrase'],
            'source_sentence': m['sentence'],
            'mitre_technique_id': m['technique_id'],
            'mitre_technique_name': m['technique_name'],
            'similarity_distance': m['distance']
        })
    
    df = pd.DataFrame(export_data)
    df.to_csv(output_path, index=False)
    print(f"✓ Results exported to: {output_path}")
    print(f"  {len(accepted)} attack phrases with MITRE mappings")

def get_summary_stats(results: Dict) -> Dict:
    """
    Get summary statistics from pipeline results.
    
    Args:
        results: Output from process_text_through_pipeline()
    
    Returns:
        Dict with summary statistics
    """
    n_sentences = results['input']['n_sentences']
    n_attack = results['step3a']['n_attack']
    n_phrases = results['step3b']['n_phrases']
    n_mapped = results['step4']['n_mapped']
    
    return {
        'total_sentences': n_sentences,
        'attack_sentences': n_attack,
        'attack_sentence_rate': n_attack / n_sentences if n_sentences > 0 else 0,
        'extracted_phrases': n_phrases,
        'phrases_per_sentence': n_phrases / n_attack if n_attack > 0 else 0,
        'mapped_techniques': n_mapped,
        'mapping_rate': n_mapped / n_phrases if n_phrases > 0 else 0,
        'unique_techniques': len(set(m['technique_id'] for m in results['step4']['accepted'])),
        'funnel_efficiency': n_mapped / n_sentences if n_sentences > 0 else 0
    }

print("✓ Display helper functions defined")

✓ Display helper functions defined


## 🧪 Section 7: Test the Pipeline

Run the complete pipeline on sample text.

### 7.1: Test with Sample CTI Report

In [53]:
# Read CTI report from file
report_path = "/home/jovyan/work/SampleReports/StuxNet.txt"

print(f"Reading report from: {report_path}")
with open(report_path, 'r', encoding='utf-8') as f:
    sample_text = f.read()

print(f"✓ Loaded report: {len(sample_text)} characters")
print(f"✓ Preview: {sample_text[:200]}...")
print()

# Run the pipeline
results = process_text_through_pipeline(sample_text, verbose=True)

Reading report from: /home/jovyan/work/SampleReports/StuxNet.txt
✓ Loaded report: 24513 characters
✓ Preview: The Stuxnet Worm
Paul Mueller and Babak Yadegari
1 Overview of Stuxnet
Stuxnet is a sophisticated worm designed to target only specific Siemens SCADA (industrial control)
systems.
It makes use of an u...


RUNNING COMPLETE LADDER PIPELINE
  1. The Stuxnet Worm
Paul Mueller and Babak Yadegari
1 Overview of Stuxnet
Stuxnet is a sophisticated worm designed to target only specific Siemens SCADA (industrial control)
systems.
  2. It makes use of an unprecedented four 0-day vulnerabilities- attacks that make use of a security
vulnerability in an application, before the vulnerability is known to the application’s developers.
  3. It also uses rootkits - advanced techniques to hide itself from users and anti-malware software -
on both Windows and the control computers it targets.
  4. It employs two stolen digital certificates to
sign its drivers, and its creators needed a large amount

### 7.2: Display Results

In [54]:
# Display formatted results
display_results(results, show_rejected=True)


FINAL RESULTS: ATTACK PHRASES WITH MITRE ATT&CK MAPPINGS

 #                                         Attack Phrase Sub-Technique                              Name Parent Tech Distance
 1 installing signed drivers on Windows operating sys...     T1553.003 SIP and Trust Provider Hijacki...       T1553    0.498
 2 communicates with command and control servers to p...     T1102.003             One-Way Communication       T1102    0.367
 3 registers code to an infected Windows computer tha...     T1052.001             Exfiltration over USB       T1052    0.493
 4 causes Windows to automatically run a file on remo...     T1547.001 Registry Run Keys / Startup Fo...       T1547    0.482
 5                 valid commands to infect the computer     T1204.002                    Malicious File       T1204    0.482
 6 attacks its database using SQL commands to upload ...     T1608.002                       Upload Tool       T1608    0.476
 7 places a dropper file on any shares on remote comp...   

### 7.3: Show Summary Statistics

In [102]:
# Get summary statistics
stats = get_summary_stats(results)

print("\n" + "="*60)
print("PIPELINE SUMMARY STATISTICS")
print("="*60)
print(f"Total sentences processed:     {stats['total_sentences']}")
print(f"Attack sentences identified:   {stats['attack_sentences']} ({stats['attack_sentence_rate']*100:.1f}%)")
print(f"Attack phrases extracted:      {stats['extracted_phrases']}")
print(f"Phrases per attack sentence:   {stats['phrases_per_sentence']:.2f}")
print(f"Phrases mapped to MITRE:       {stats['mapped_techniques']} ({stats['mapping_rate']*100:.1f}%)")
print(f"Unique MITRE techniques:       {stats['unique_techniques']}")
print(f"Overall funnel efficiency:     {stats['funnel_efficiency']*100:.1f}%")
print("="*60 + "\n")


PIPELINE SUMMARY STATISTICS
Total sentences processed:     197
Attack sentences identified:   43 (21.8%)
Attack phrases extracted:      69
Phrases per attack sentence:   1.60
Phrases mapped to MITRE:       46 (66.7%)
Unique MITRE techniques:       27
Overall funnel efficiency:     23.4%



### 7.4: Export Results to CSV

## 📈 Section 10: Visualize Pipeline Flow

In [104]:
def visualize_pipeline_flow(results: Dict):
    """
    Visualize how data flows through the pipeline with ASCII art.
    
    Args:
        results: Output from process_text_through_pipeline()
    """
    stats = get_summary_stats(results)
    
    print("\n" + "="*70)
    print("PIPELINE DATA FLOW VISUALIZATION")
    print("="*70)
    print()
    print("  ┌─────────────────────────────────────────┐")
    print(f"  │  📄 INPUT TEXT                          │")
    print(f"  │  {stats['total_sentences']:3d} sentences                           │")
    print("  └─────────────────────────────────────────┘")
    print("                    │")
    print("                    ▼")
    print("  ┌─────────────────────────────────────────┐")
    print("  │  🔍 STEP 3a: Sentence Classification   │")
    print(f"  │  {stats['attack_sentences']:3d} attack sentences ({stats['attack_sentence_rate']*100:5.1f}%)       │")
    print(f"  │  {stats['total_sentences']-stats['attack_sentences']:3d} non-attack filtered out          │")
    print("  └─────────────────────────────────────────┘")
    print("                    │")
    print("                    ▼")
    print("  ┌─────────────────────────────────────────┐")
    print("  │  🎯 STEP 3b: Phrase Extraction         │")
    print(f"  │  {stats['extracted_phrases']:3d} attack phrases                    │")
    print(f"  │  ({stats['phrases_per_sentence']:.2f} phrases/sentence avg)            │")
    print("  └─────────────────────────────────────────┘")
    print("                    │")
    print("                    ▼")
    print("  ┌─────────────────────────────────────────┐")
    print("  │  🗺️  STEP 4: MITRE Mapping              │")
    print(f"  │  {stats['mapped_techniques']:3d} phrases mapped ({stats['mapping_rate']*100:5.1f}%)         │")
    print(f"  │  {stats['unique_techniques']:3d} unique MITRE techniques           │")
    print("  └─────────────────────────────────────────┘")
    print("                    │")
    print("                    ▼")
    print("  ┌─────────────────────────────────────────┐")
    print(f"  │  ✅ FINAL OUTPUT                        │")
    print(f"  │  {stats['mapped_techniques']:3d} attack patterns with MITRE IDs    │")
    print(f"  │  Overall efficiency: {stats['funnel_efficiency']*100:5.1f}%            │")
    print("  └─────────────────────────────────────────┘")
    print()
    print("="*70 + "\n")

# Visualize the flow from our test
visualize_pipeline_flow(results)


PIPELINE DATA FLOW VISUALIZATION

  ┌─────────────────────────────────────────┐
  │  📄 INPUT TEXT                          │
  │  197 sentences                           │
  └─────────────────────────────────────────┘
                    │
                    ▼
  ┌─────────────────────────────────────────┐
  │  🔍 STEP 3a: Sentence Classification   │
  │   43 attack sentences ( 21.8%)       │
  │  154 non-attack filtered out          │
  └─────────────────────────────────────────┘
                    │
                    ▼
  ┌─────────────────────────────────────────┐
  │  🎯 STEP 3b: Phrase Extraction         │
  │   69 attack phrases                    │
  │  (1.60 phrases/sentence avg)            │
  └─────────────────────────────────────────┘
                    │
                    ▼
  ┌─────────────────────────────────────────┐
  │  🗺️  STEP 4: MITRE Mapping              │
  │   46 phrases mapped ( 66.7%)         │
  │   27 unique MITRE techniques           │
  └────────────────

## 🎓 Section 11: Usage Notes and Best Practices

### Key Points:

1. **Model Loading**: Make sure your model paths in the configuration are correct

2. **Step 3a**: Filters sentences to keep only attack-relevant ones
   - Adjust `step3a_confidence_threshold` to control sensitivity
   - Lower threshold = more sentences pass through (higher recall, lower precision)

3. **Step 3b**: Extracts specific attack phrases using NER
   - Verb filtering removes phrases without action words (LADDER post-processing)
   - Set `step3b_filter_verbs=False` to disable this filter

4. **Step 4**: Maps phrases to MITRE using semantic similarity
   - `mitre_threshold=0.6` is optimal from LADDER paper
   - `mitre_weight_title=0.4` balances title vs description similarity
   - Lower threshold = more aggressive matching (more mappings, possibly less accurate)

5. **Performance**: 
   - First run will be slow (model loading + embedding computation)
   - Subsequent runs are faster (embeddings are cached)
   - GPU significantly speeds up Steps 3a and 3b

6. **Troubleshooting**:
   - If no results: Check that models are loaded correctly
   - If too few mappings: Lower `mitre_threshold` or check input quality
   - If too many false positives: Increase `step3a_confidence_threshold`

## 💾 Section 12: Save Pipeline State (Optional)

## ✅ Pipeline Ready!

You now have a complete end-to-end LADDER pipeline. To use it:

1. **Update paths** in Section 2 (Configuration)
2. **Run Sections 1-4** to load models and define functions  
3. **Use Section 7** to test with sample text
4. **Use Section 8** for your own CTI reports
5. **Use Section 9** for batch processing multiple reports

The pipeline automatically:
- Filters non-attack sentences (Step 3a)
- Extracts attack phrases (Step 3b)
- Maps to MITRE ATT&CK (Step 4)
- Displays results in formatted tables
- Exports to CSV for further analysis

In [110]:
# T1055 Process Injection NER Refinement
# Add this to your End-to-End LADDER Pipeline

"""
This module adds a specialized refinement step for T1055 (Process Injection) detections.
When any phrase maps to a T1055 sub-technique, we run a specialized NER model to extract
fine-grained entities like:
- B-PROC/I-PROC: Malicious processes
- B-TARG/I-TARG: Target libraries/processes
- B-TTP/I-TTP: TTP arguments
"""

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from typing import List, Dict, Tuple

# ============================================================================
# SECTION 1: Load T1055 NER Model
# ============================================================================

class T1055NERModel:
    """Specialized NER model for T1055 Process Injection detection"""
    
    def __init__(self, model_checkpoint: str):
        """
        Initialize the T1055 NER model.
        
        Args:
            model_checkpoint: Path to trained XLM-RoBERTa-large model
        """
        print(f"Loading T1055 NER model from: {model_checkpoint}")
        
        # Load tokenizer with add_prefix_space for RoBERTa
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_checkpoint, 
            add_prefix_space=True
        )
        
        # Load model
        self.model = AutoModelForTokenClassification.from_pretrained(
            model_checkpoint
        )
        
        # Get label mappings
        self.id2label = self.model.config.id2label
        self.label2id = self.model.config.label2id
        self.label_names = [self.id2label[i] for i in range(len(self.id2label))]
        
        # Move to GPU if available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        
        print(f"✓ T1055 NER model loaded")
        print(f"  Device: {self.device}")
        print(f"  Labels: {self.label_names}")
    
    def extract_entities(self, sentence: str) -> List[Dict]:
        """
        Extract T1055-specific entities from a sentence.
        
        Args:
            sentence: Input sentence (should be related to T1055)
        
        Returns:
            List of entity dicts with {text, label, start, end}
        """
        # Tokenize
        tokens = sentence.split()
        
        if not tokens:
            return []
        
        # Encode
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        # Get word_ids BEFORE moving to device
        word_ids = encoding.word_ids()
        
        # Move to device
        encoding_tensors = {k: v.to(self.device) for k, v in encoding.items()}
        
        # Predict
        with torch.no_grad():
            outputs = self.model(**encoding_tensors)
            predictions = outputs.logits.argmax(-1)[0].cpu().tolist()
        
        # Align predictions with words (take first subword only)
        word_predictions = {}
        current_word = None
        
        for token_idx, word_id in enumerate(word_ids):
            if word_id is None:
                continue
            
            # Only take first subword prediction for each word
            if word_id != current_word:
                current_word = word_id
                pred_label = self.label_names[predictions[token_idx]]
                word_predictions[word_id] = pred_label
        
        # Extract entities from word predictions
        entities = []
        current_entity = None
        current_tokens = []
        
        for word_idx in sorted(word_predictions.keys()):
            label = word_predictions[word_idx]
            token = tokens[word_idx]
            
            if label.startswith('B-'):
                # Start of new entity
                if current_entity:
                    # Save previous entity
                    entities.append({
                        'text': ' '.join(current_tokens),
                        'label': current_entity,
                        'tokens': current_tokens.copy()
                    })
                
                current_entity = label[2:]  # Remove 'B-'
                current_tokens = [token]
                
            elif label.startswith('I-') and current_entity:
                # Continuation of current entity
                entity_type = label[2:]  # Remove 'I-'
                if entity_type == current_entity:
                    current_tokens.append(token)
                else:
                    # Entity type changed, save previous and start new
                    entities.append({
                        'text': ' '.join(current_tokens),
                        'label': current_entity,
                        'tokens': current_tokens.copy()
                    })
                    current_entity = entity_type
                    current_tokens = [token]
            
            else:
                # 'O' tag or mismatched I- tag
                if current_entity:
                    # Save previous entity
                    entities.append({
                        'text': ' '.join(current_tokens),
                        'label': current_entity,
                        'tokens': current_tokens.copy()
                    })
                    current_entity = None
                    current_tokens = []
        
        # Don't forget last entity
        if current_entity:
            entities.append({
                'text': ' '.join(current_tokens),
                'label': current_entity,
                'tokens': current_tokens.copy()
            })
        
        return entities


# ============================================================================
# SECTION 2: T1055 Detection and Refinement
# ============================================================================

def check_t1055_detection(mapped_results: List[Dict]) -> bool:
    """
    Check if any mapped result is a T1055 sub-technique.
    
    Args:
        mapped_results: Results from Step 4 mapping
    
    Returns:
        True if any T1055 sub-technique detected
    """
    for result in mapped_results:
        if result['accepted'] and result['technique_id'] == 'T1055':
            return True
    return False


def refine_t1055_detections(
    mapped_results: List[Dict],
    ner_model: T1055NERModel,
    verbose: bool = True
) -> List[Dict]:
    """
    Refine T1055 detections by extracting fine-grained entities.
    
    Args:
        mapped_results: Results from Step 4 mapping
        ner_model: Loaded T1055 NER model
        verbose: Print progress
    
    Returns:
        List of refined T1055 detections with extracted entities
    """
    t1055_refinements = []
    
    for result in mapped_results:
        # Only process accepted T1055 mappings
        if not result['accepted'] or result['technique_id'] != 'T1055':
            continue
        
        if verbose:
            print(f"\\n🔍 Refining T1055 detection:")
            print(f"  Phrase: {result['phrase']}")
            print(f"  Sub-technique: {result['subtech_id']} - {result['subtech_name']}")
        
        # Extract entities from the source sentence
        sentence = result['sentence']
        entities = ner_model.extract_entities(sentence)
        
        # Organize entities by type
        entity_types = {}
        for ent in entities:
            label = ent['label']
            if label not in entity_types:
                entity_types[label] = []
            entity_types[label].append(ent['text'])
        
        refinement = {
            'phrase': result['phrase'],
            'sentence': sentence,
            'subtech_id': result['subtech_id'],
            'subtech_name': result['subtech_name'],
            'distance': result['distance'],
            'entities': entities,
            'entity_summary': entity_types
        }
        
        if verbose:
            print(f"  Entities found: {len(entities)}")
            for label, texts in entity_types.items():
                print(f"    {label}: {', '.join(texts)}")
        
        t1055_refinements.append(refinement)
    
    return t1055_refinements


# ============================================================================
# SECTION 3: Integration with Pipeline
# ============================================================================

def add_t1055_refinement_to_pipeline(
    results: Dict,
    ner_model_path: str,
    verbose: bool = True
) -> Dict:
    """
    Add T1055 refinement step to pipeline results.
    
    This is the main function to integrate into your pipeline.
    Call it after Step 4 mapping completes.
    
    Args:
        results: Output from process_text_through_pipeline()
        ner_model_path: Path to trained T1055 NER model
        verbose: Print progress
    
    Returns:
        Updated results dict with 't1055_refinement' section
    """
    if verbose:
        print("\\n" + "="*70)
        print("T1055 PROCESS INJECTION REFINEMENT")
        print("="*70)
    
    # Get mapped results from Step 4
    mapped = results['step4']['mapped']
    
    # Check if any T1055 detections
    has_t1055 = check_t1055_detection(mapped)
    
    if not has_t1055:
        if verbose:
            print("\\n⚠️  No T1055 (Process Injection) techniques detected.")
            print("   Skipping NER refinement.")
        
        results['t1055_refinement'] = {
            'enabled': False,
            'detections': [],
            'n_detections': 0
        }
        return results
    
    # Load NER model
    if verbose:
        print(f"\\n✓ T1055 detections found! Loading specialized NER model...")
    
    ner_model = T1055NERModel(ner_model_path)
    
    # Run refinement
    refinements = refine_t1055_detections(mapped, ner_model, verbose=verbose)
    
    # Add to results
    results['t1055_refinement'] = {
        'enabled': True,
        'detections': refinements,
        'n_detections': len(refinements)
    }
    
    if verbose:
        print("\\n" + "="*70)
        print(f"✓ T1055 refinement complete!")
        print(f"  Detected: {len(refinements)} T1055 instances")
        print(f"  Total entities extracted: {sum(len(r['entities']) for r in refinements)}")
        print("="*70 + "\\n")
    
    return results


# ============================================================================
# SECTION 4: Display Functions
# ============================================================================

def display_t1055_refinements(results: Dict):
    """
    Display T1055 refinement results in a formatted way.
    
    Args:
        results: Pipeline results with t1055_refinement section
    """
    if 't1055_refinement' not in results:
        print("⚠️  No T1055 refinement data available.")
        return
    
    refinement_data = results['t1055_refinement']
    
    if not refinement_data['enabled']:
        print("ℹ️  T1055 refinement was not enabled (no T1055 detections).")
        return
    
    detections = refinement_data['detections']
    
    if not detections:
        print("⚠️  No T1055 detections to display.")
        return
    
    print("\\n" + "="*90)
    print("T1055 PROCESS INJECTION - DETAILED ENTITY EXTRACTION")
    print("="*90)
    
    for i, det in enumerate(detections, 1):
        print(f"\\n{'='*90}")
        print(f"DETECTION #{i}")
        print(f"{'='*90}")
        print(f"Sub-Technique: {det['subtech_id']} - {det['subtech_name']}")
        print(f"Confidence: {det['distance']:.3f}")
        print(f"\\nAttack Phrase: {det['phrase']}")
        print(f"\\nSource Sentence:")
        print(f"  {det['sentence']}")
        
        if det['entities']:
            print(f"\\nExtracted Entities ({len(det['entities'])} total):")
            print("-" * 90)
            
            # Group by entity type
            entity_summary = det['entity_summary']
            
            if 'PROC' in entity_summary:
                print(f"\\n  🔴 MALICIOUS PROCESSES:")
                for proc in entity_summary['PROC']:
                    print(f"     • {proc}")
            
            if 'TARG' in entity_summary:
                print(f"\\n  🎯 TARGET LIBRARIES/PROCESSES:")
                for targ in entity_summary['TARG']:
                    print(f"     • {targ}")
            
            if 'TTP' in entity_summary:
                print(f"\\n  ⚙️  TTP ARGUMENTS:")
                for ttp in entity_summary['TTP']:
                    print(f"     • {ttp}")
            
            # Show raw entity list
            print(f"\\n  📋 All Entities (with labels):")
            for ent in det['entities']:
                print(f"     [{ent['label']}] {ent['text']}")
        else:
            print(f"\\n  ⚠️  No entities extracted from this sentence.")
    
    print("\\n" + "="*90 + "\\n")


def export_t1055_refinements_to_csv(results: Dict, output_path: str = "t1055_entities.csv"):
    """
    Export T1055 refinement results to CSV.
    
    Args:
        results: Pipeline results with t1055_refinement section
        output_path: Path to save CSV
    """
    if 't1055_refinement' not in results or not results['t1055_refinement']['enabled']:
        print("⚠️  No T1055 refinement data to export.")
        return
    
    detections = results['t1055_refinement']['detections']
    
    if not detections:
        print("⚠️  No T1055 detections to export.")
        return
    
    # Flatten entities into rows
    export_data = []
    
    for det in detections:
        for ent in det['entities']:
            export_data.append({
                'subtech_id': det['subtech_id'],
                'subtech_name': det['subtech_name'],
                'attack_phrase': det['phrase'],
                'sentence': det['sentence'],
                'entity_text': ent['text'],
                'entity_label': ent['label'],
                'distance': det['distance']
            })
    
    if export_data:
        df = pd.DataFrame(export_data)
        df.to_csv(output_path, index=False)
        print(f"✓ T1055 entities exported to: {output_path}")
        print(f"  {len(export_data)} entities from {len(detections)} T1055 detections")
    else:
        print("⚠️  No entities to export.")


# ============================================================================
# SECTION 5: Usage Example
# ============================================================================

"""
USAGE IN YOUR PIPELINE:
"""



# Add T1055 refinement:
ner_model_path = "/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480"
results = add_t1055_refinement_to_pipeline(results, ner_model_path, verbose=True)

# Display refined results:
display_t1055_refinements(results)

# Export to CSV:
#export_t1055_refinements_to_csv(results, "t1055_entities.csv")

print("✓ T1055 NER refinement module loaded")

\n======================================================================
T1055 PROCESS INJECTION REFINEMENT
\n✓ T1055 detections found! Loading specialized NER model...
Loading T1055 NER model from: /home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480


The tokenizer you are loading from '/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed1/checkpoint-13480' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✓ T1055 NER model loaded
  Device: cuda
  Labels: ['B-MalProcess', 'B-TargetProcess', 'B-MalLibrary', 'B-TargetLibrary', 'B-TTPArgument', 'I-MalProcess', 'I-TargetProcess', 'I-MalLibrary', 'I-TargetLibrary', 'I-TTPArgument', 'O']
\n🔍 Refining T1055 detection:
  Phrase: copies itself, places the copy on remote computers via this vulnerability,
  Sub-technique: T1055.013 - Process Doppelgänging
  Entities found: 0
\n🔍 Refining T1055 detection:
  Phrase: inject it into a chosen process - add its own code into a running process,
  Sub-technique: T1055.002 - Portable Executable Injection
  Entities found: 0
\n======================================================================
✓ T1055 refinement complete!
  Detected: 2 T1055 instances
  Total entities extracted: 0
======================================================================\n
\n==========================================================================================
T1055 PROCESS INJECTION - DETAILED ENTITY EXTRACTION
\n=======

In [85]:
# ============================================================================
# SECTION 6: Full Report Processing and Comparison
# ============================================================================
#CHECK - T1055 NOTEBOOK IN CODE 

✓ Full report processing and comparison functions loaded
